# Experiment 04: Agent ReAct Core, Planning Protocol & Step Tracing

**Subsystem Goal:** Benchmark and validate the native Autonomous ReAct Agent Loop:
1. **Dual-Schema Tool Architecture:**
   - **8 Dynamic Domain Tools:** Exposed to the LLM by the FastMCP server (`indic-transcribe`, `indic-translate`, `indic-speak`, `indic-ocr`, `inspect_media`, `read_text_file`, `write_text_file`, `detect_language`). `get_cache_stats` and `clear_cache` stay registered for operators but are excluded from the model catalog.
   - **2 Structural Protocol Tools:** `submit_plan` (forced on Turn 0) and `finish` (forced exit tool).
2. **Structured Planning Step (Turn 0):** Enforce deliberate analytical reasoning before tool invocation — assessing feasibility, setting goals, documenting explicit assumptions, and establishing step sequences.
3. **Structured Finish & Verification:** Replace ambiguous textual responses with explicit `finish` tool calls validating generated artifacts, execution status (`completed`, `impossible`, `failed`), and maintained assumptions.
4. **Real-Time Step Tracing:** Every ReAct action and observation is flushed immediately to `trace.jsonl` with latency, rupee spend, and status.
5. **Dual Reporting Invariant:** Structured machine-readable `report.json` and human-readable `result.md` produced for every run.


In [ ]:
import sys
import os
import json
import asyncio
import logging
from pathlib import Path

# Robust recipe root resolution
for base in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (base / "sample_data" / "demo1_indic_nlp_paper.pdf").is_file():
        recipe_root = base
        break
    cand = base / "ref" / "Bodhan-AI-Cookbook" / "examples" / "bodhan_tool_agent"
    if (cand / "sample_data" / "demo1_indic_nlp_paper.pdf").is_file():
        recipe_root = cand
        break
    cand2 = base / "examples" / "bodhan_tool_agent"
    if (cand2 / "sample_data" / "demo1_indic_nlp_paper.pdf").is_file():
        recipe_root = cand2
        break
else:
    recipe_root = Path.cwd().resolve()

if str(recipe_root) not in sys.path:
    sys.path.insert(0, str(recipe_root))

from src.bodhan_agent.agent import (
    _get_openai_tool_definitions,
    run_task_agent,
    SUBMIT_PLAN_TOOL,
    FINISH_TOOL,
)
from src.bodhan_agent.trace import TraceLogger
from src.common.config import BodhanConfig

# Silence per-request httpx INFO spam
logging.getLogger("httpx").setLevel(logging.ERROR)

print("✓ Agent ReAct core, planning & finish schemas, and tracing logger loaded successfully!")
print(f"✓ Recipe Root: {recipe_root}")


## 1. Inspect OpenAI Tool Calling Schemas & Structural Protocol Tools

Verify that FastMCP tools translate cleanly into standard OpenAI tool definitions, and inspect the structural `submit_plan` and `finish` protocol schemas.


In [ ]:
tools = _get_openai_tool_definitions()
print(f"Generated {len(tools)} OpenAI domain tool schemas from FastMCP:\n")
for t in tools:
    fn = t["function"]
    print(f"- {fn['name']:<24} : {fn['description'][:70]}...")
    print(f"  Parameters: {list(fn['parameters'].get('properties', {}).keys())}")

assert len(tools) == 8, f"Expected 8 model-facing tools, got {len(tools)}"
print(f"\n✓ All {len(tools)} FastMCP domain tools successfully mapped to OpenAI function schemas!")

print("\n" + "=" * 75)
print("              STRUCTURAL AGENT PROTOCOL TOOLS (PLAN & FINISH)            ")
print("=" * 75)
for p_tool in (SUBMIT_PLAN_TOOL, FINISH_TOOL):
    fn = p_tool["function"]
    print(f"- {fn['name']:<14} : {fn['description']}")
    print(f"  Required Fields : {fn['parameters']['required']}")
    print(f"  All Parameters  : {list(fn['parameters']['properties'].keys())}\n")

assert SUBMIT_PLAN_TOOL["function"]["name"] == "submit_plan"
assert FINISH_TOOL["function"]["name"] == "finish"
print("✓ Structural protocol tools (submit_plan and finish) verified!")


## 2. Test Real-Time Step Tracing

Test that `TraceLogger` creates `trace.jsonl` and flushes each step immediately to disk.

In [ ]:
test_dir = Path("outputs/exp04_test")
logger = TraceLogger(test_dir)

trace_file = test_dir / "trace.jsonl"
lines_before = (
    len(trace_file.read_text(encoding="utf-8").strip().splitlines())
    if trace_file.exists()
    else 0
)

# Step 0: Structured Planning
logger.log_step(
    step=0,
    plan="Evaluate feasibility and define execution steps",
    tool="submit_plan",
    arguments={
        "goal": "Detect media type and transcribe",
        "feasible": True,
        "assumptions": ["Audio is valid WAV 24kHz"],
        "steps": ["inspect_media", "transcribe_audio"],
    },
    result_summary="Plan approved: 2 steps",
    latency_ms=850,
    cost_inr=0.0,
    cumulative_cost_inr=0.0,
)

# Step 1: Execution Action
logger.log_step(
    step=1,
    plan="Detect media type from input clip",
    tool="inspect_media",
    arguments={"file_path": "sample.wav"},
    result_summary="Audio WAV 24kHz mono, 14.2 seconds",
    latency_ms=115,
    cost_inr=0.0,
    cumulative_cost_inr=0.0,
)

# Step 2: Finish Action
logger.log_step(
    step=2,
    plan="Conclude task execution with finish",
    tool="finish",
    arguments={
        "status": "completed",
        "answer": "Audio inspected and transcribed successfully.",
        "artifacts": ["sample_transcript.md"],
    },
    result_summary="Task concluded: completed",
    latency_ms=520,
    cost_inr=0.0,
    cumulative_cost_inr=0.0,
)

assert trace_file.exists(), "trace.jsonl was not created on disk!"
lines_after = len(trace_file.read_text(encoding="utf-8").strip().splitlines())
print(f"trace.jsonl lines: {lines_before} -> {lines_after} (flush verified)")
assert lines_after == lines_before + 3, "Exactly 3 new step lines (Plan, Act, Finish) must be flushed!"
print("✓ Step tracing with Step 0 Planning and Finish verified!")


## 3. Runtime Config & Key Sanity Check

`run_task_agent` builds an OpenAI-compatible reasoning client from `LLM_BASE_URL` /
`LLM_MODEL` / `LLM_API_KEY`, and the Bodhan tools authenticate with their own keys. A
missing key means every reasoning or paid call fails mid-loop, so verify the
environment before spending rupees.

In [ ]:
import os

# Some OpenAI-compatible endpoints (e.g. gpt-6-luna on /v1/chat/completions)
# require an explicit reasoning_effort before they accept function tools.
# Default to 'none' unless the environment already specifies a level;
# run_task_agent re-reads the environment on every call.
os.environ.setdefault("LLM_REASONING_EFFORT", "none")

config = BodhanConfig.from_env()

print(f"LLM model:             {config.llm_model}")
print(f"LLM base URL:          {config.llm_base_url or '(default: api.openai.com)'}")
print(f"LLM reasoning effort:  {config.llm_reasoning_effort or '(unset)'}")
print(f"LLM API key set:       {bool(config.llm_api_key)}")
print(f"Transcribe key set:    {bool(config.transcribe_key)}")
print(f"Translate key set:     {bool(config.translate_key)}")
print(f"Speak key set:         {bool(config.speak_key)}")
print(f"OCR key set:           {bool(config.ocr_key)}")

assert config.llm_api_key, "LLM_API_KEY missing: the ReAct reasoning loop cannot run!"
print("\nConfig ready: the ReAct loop can reach its reasoning LLM and paid Bodhan tools.")

## 4. Ask the Agent #1: Free-Tool Media Inventory (₹0)

Point the agent at the `sample_data` directory and ask what is on disk. This exercises
the full ReAct cycle (Plan -> Act -> Observe -> Reflect) using only free tools:
`inspect_media`, `read_text_file`, `detect_language`, and `write_text_file`. Expected:
the agent inventories each file, writes `artifacts/media_inventory.md`, verifies it,
then reports.

Jupyter top-level `await` drives the async ReAct loop directly (same pattern as
Experiment 06).

In [ ]:
run1_dir = Path("outputs/exp04_run_inventory")

result1 = await run_task_agent(
    task=(
        "Look at the files in the input directory, inspect each one with inspect_media, "
        "and write a short English inventory (media type, duration/size) to "
        "artifacts/media_inventory.md. Then report what you found."
    ),
    files_dir=str(recipe_root / "sample_data"),
    out_dir=run1_dir,
    max_steps=12,
    max_cost_inr=10.0,
)

print(f"Status:     {result1['status']}")
print(f"Steps:      {result1['total_steps']}")
print(f"Total cost: ₹{result1['total_cost_inr']:.2f}")
print(f"Output dir: {result1['output_dir']}")
print(f"\nFinal answer:\n{result1['final_answer']}")

## 5. Trace Viewer: Every ReAct Step on Record

`TraceLogger` flushes each step to `trace.jsonl` the moment it happens. The helper
below renders any run's step table: the plan the agent stated, the tool it acted
with, latency, rupee cost, and status. It is reused by every run below.

In [ ]:
import json

def show_trace(run_dir: Path) -> None:
    trace_path = Path(run_dir) / "trace.jsonl"
    if not trace_path.exists():
        print(f"  SKIP: {trace_path} not found - run has not been executed yet.")
        return
    records = [
        json.loads(line)
        for line in trace_path.read_text(encoding="utf-8").strip().splitlines()
        if line.strip()
    ]
    print(f"=================================================================")
    print(f"              TRACE AUDIT: {Path(run_dir).name} ({len(records)} steps)")
    print(f"=================================================================")
    for rec in records:
        step_num = rec['step']
        tag = "[PLAN]  " if step_num == 0 else ("[FINISH]" if rec['tool'] == 'finish' else f"[{step_num:>2}]    ")
        status_str = rec['status'].upper()
        tool_str = rec['tool']
        lat_str = f"{rec['latency_ms']} ms"
        cost_str = f"₹{rec['cost_inr']:.4f}"
        cum_str = f"Σ₹{rec['cumulative_cost_inr']:.4f}"
        
        print(f"{tag} {status_str:<9} {tool_str:<22} {lat_str:>8}  {cost_str:>8} ({cum_str:>9})")
        print(f"       plan: {rec.get('plan', '')[:95]}")
        print(f"       got : {rec.get('result_summary', '')[:95]}\n")
    print("=" * 65)

show_trace(run1_dir)


## 6. Ask the Agent #2: Paid Translation Chain

A multi-step task that spends rupees: find the plain-text Bengali verses among the
inputs, detect the language, translate to English, and save artifacts. Watch the
ReAct chain `read_text_file -> detect_language -> translate_text -> write_text_file`
unfold with real `CostLedger` accounting.

In [ ]:
run2_dir = Path("outputs/exp04_run_translate")

result2 = await run_task_agent(
    task=(
        "Among the input files there is one plain-text file of Bengali verses. "
        "Read it, detect its language, translate the verses into English, and save "
        "the English translation to artifacts/gitanjali_english.md with a "
        "one-paragraph English summary at the top. Report which language you detected."
    ),
    files_dir=str(recipe_root / "sample_data"),
    out_dir=run2_dir,
    max_steps=10,
    max_cost_inr=10.0,
)

print(f"Status:     {result2['status']}")
print(f"Steps:      {result2['total_steps']}")
print(f"Total cost: ₹{result2['total_cost_inr']:.2f}")
print(f"\nFinal answer:\n{result2['final_answer']}\n")

show_trace(run2_dir)

## 7. Ask the Agent #3: Corrupted Audio (Honest Failure)

Corrupted input must never produce fabricated output.
`fail_corrupted_audio_header.wav` has a broken WAV header, so `transcribe_audio`
raises a structured `ToolError`. The agent should observe the error, refuse to
invent a transcript, and explain honestly - visible as `status: error` lines in
the trace. The run is scoped to the single corrupted file so the agent cannot
quietly transcribe a healthy one instead.

In [ ]:
run3_dir = Path("outputs/exp04_run_corrupt_audio")

result3 = await run_task_agent(
    task=(
        "Transcribe the audio file in the input directory and translate the "
        "transcript into Hindi. If anything is wrong with the file, explain "
        "instead of guessing."
    ),
    files_dir=str(recipe_root / "sample_data" / "fail_corrupted_audio_header.wav"),
    out_dir=run3_dir,
    max_steps=6,
    max_cost_inr=10.0,
)

print(f"Status:     {result3['status']}")
print(f"Steps:      {result3['total_steps']}")
print(f"Total cost: ₹{result3['total_cost_inr']:.2f}")
print(f"\nFinal answer:\n{result3['final_answer']}\n")

show_trace(run3_dir)

trace3 = [
    json.loads(line)
    for line in (run3_dir / "trace.jsonl").read_text(encoding="utf-8").strip().splitlines()
    if line.strip()
]
error_steps = [rec for rec in trace3 if rec["status"] == "error"]
if error_steps:
    print(f"\n{len(error_steps)} error step(s) recorded - the agent observed the failure instead of fabricating.")
else:
    print("\nNo error steps: the agent detected the corruption earlier (e.g. via inspect_media) and halted honestly.")

## 8. Report Compilation: report.json + result.md

Every run must compile its structured report: `report.json` (machine-readable) and
`result.md` (human-readable, linking generated artifacts). Verify the output-layout
invariant across all three runs above.

In [ ]:
for run_dir in (run1_dir, run2_dir, run3_dir):
    print(f"=== {run_dir.name} ===")
    if not Path(run_dir).exists():
        print("  SKIP: run directory not found - this run has not been executed yet.")
        continue
    missing = [f for f in ("trace.jsonl", "report.json", "result.md") if not (Path(run_dir) / f).exists()]
    if missing:
        print(f"  SKIP: missing {missing} - run did not complete.")
        continue
    artifacts = sorted(p.name for p in (Path(run_dir) / "artifacts").iterdir() if p.is_file())
    print(f"  artifacts/: {artifacts if artifacts else '(none)'}")
    report = json.loads((Path(run_dir) / "report.json").read_text(encoding="utf-8"))
    print(
        f"  status={report['status']}  steps={report['total_steps']}  "
        f"cost=\u20b9{report['total_cost_inr']:.2f}  elapsed={report['elapsed_seconds']}s"
    )

print("\nOutput-layout check complete (executed runs must include trace.jsonl, report.json, result.md, artifacts/).")

if (run2_dir / "result.md").exists():
    md_text = (run2_dir / "result.md").read_text(encoding="utf-8")
    print("\n--- result.md (translation run, first 1500 chars) ---")
    print(md_text[:1500])
else:
    print("\n(result.md for the translation run will print once that run has been executed.)")

## 9. Deterministic ReAct Walk-Through (Planning Step -> Action -> Finish)

To guarantee 100% reproducible verification of the complete ReAct state machine — including **Turn 0 Planning**, **Tool Execution**, **Finish Verification**, and **Artifact Audit** — even in network-isolated environments or without active OpenAI API keys, we test the complete loop with a deterministic Mock LLM runner.


In [ ]:
from unittest.mock import MagicMock, patch

class MockChoice:
    def __init__(self, message):
        self.message = message

class MockFunction:
    def __init__(self, name: str, arguments: str):
        self.name = name
        self.arguments = arguments

class MockToolCall:
    def __init__(self, id: str, name: str, arguments: dict):
        self.id = id
        self.type = 'function'
        self.function = MockFunction(name, json.dumps(arguments))

class MockMessage:
    def __init__(self, content=None, tool_calls=None):
        self.role = 'assistant'
        self.content = content
        self.tool_calls = tool_calls or []

class MockChatCompletion:
    def __init__(self, message):
        self.choices = [MockChoice(message)]

mock_run_dir = Path('outputs/exp04_run_mock_walkthrough')

# Step 0: Structured Planning Turn
msg_plan = MockMessage(
    content='Analyzing Bengali Gitanjali task feasibility.',
    tool_calls=[
        MockToolCall(
            id='call_plan_001',
            name='submit_plan',
            arguments={
                'goal': 'Read Bengali verses, translate to English, and save report',
                'feasible': True,
                'assumptions': ['Input verses are authentic Rabindranath Tagore Bengali poems'],
                'steps': ['read_text_file', 'translate_text', 'write_text_file', 'finish'],
                'target_language': 'en',
            }
        )
    ]
)

# Step 1: Tool Execution (write_text_file)
msg_step1 = MockMessage(
    content='Saving translated English verses to artifact.',
    tool_calls=[
        MockToolCall(
            id='call_tool_001',
            name='write_text_file',
            arguments={
                'file_path': 'gitanjali_english.md',
                'content': '# Gitanjali English Translation\n\nThou hast made me endless, such is thy pleasure.',
            }
        )
    ]
)

# Step 2: Finish Execution
msg_step2 = MockMessage(
    content='Task complete. English translation created and verified on disk.',
    tool_calls=[
        MockToolCall(
            id='call_finish_001',
            name='finish',
            arguments={
                'status': 'completed',
                'answer': 'Translated Bengali verses to English and saved to gitanjali_english.md.',
                'artifacts': ['gitanjali_english.md'],
                'assumptions': ['Input verses are authentic Rabindranath Tagore Bengali poems'],
            }
        )
    ]
)

mock_client = MagicMock()
mock_client.chat.completions.create.side_effect = [
    MockChatCompletion(msg_plan),
    MockChatCompletion(msg_step1),
    MockChatCompletion(msg_step2),
]

with (
    patch('src.bodhan_agent.agent.OpenAI', return_value=mock_client),
    patch.object(BodhanConfig, 'from_env', return_value=BodhanConfig(
        transcribe_key='mock-key',
        translate_key='mock-key',
        speak_key='mock-key',
        ocr_key='mock-key',
        llm_api_key='mock-key',
    )),
):
    mock_result = await run_task_agent(
        task='Read Bengali verses, translate to English, and save to artifacts/gitanjali_english.md',
        files_dir=str(recipe_root / 'sample_data'),
        out_dir=mock_run_dir,
        max_steps=5,
    )

print(f"Status:     {mock_result['status']}")
print(f"Steps:      {mock_result['total_steps']}")
print(f"Total Cost: ₹{mock_result['total_cost_inr']:.2f}")
print(f"Final Answer: {mock_result['final_answer']}\n")

show_trace(mock_run_dir)

# Validate Output Artifacts and Report Invariants
assert mock_result['status'] == 'completed'
assert (mock_run_dir / 'report.json').exists()
assert (mock_run_dir / 'result.md').exists()
assert (mock_run_dir / 'trace.jsonl').exists()
assert (mock_run_dir / 'artifacts' / 'gitanjali_english.md').exists()

report_data = json.loads((mock_run_dir / 'report.json').read_text(encoding='utf-8'))
assert report_data['plan']['feasible'] is True
assert len(report_data['plan']['steps']) == 4
assert report_data['status'] == 'completed'
assert report_data['verification']['verified'] is True
print('✓ Deterministic ReAct loop verified: Planning (Turn 0) -> Action -> Finish -> Verification!')
